# 1 · Preparation and data quality
Weekly store x product sales from dunnhumby *Breakfast at the Frat* (REAL RETAIL DATA, not redistributed). `src/prepare.py` cleans it and tags promotions (BR-02) and quality issues (BR-07); `sql/` builds the baseline and aggregates in DuckDB.

In [1]:
import os, sys
from pathlib import Path
# run from the repository root so relative paths work
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import pandas as pd, numpy as np
pd.set_option("display.width", 160, "display.max_columns", 20)

## Hierarchy: category > manufacturer > product (last 52 weeks, from `sql/03_hierarchy_aggregates.sql`)

In [2]:
h = pd.read_csv('data/output/sql_03_hierarchy_aggregates.csv')
h[h['level_id'] >= 1]

,category,manufacturer,product,level_id,units,revenue,avg_price,avg_base_price,promo_revenue_share
0,All categories,(all),(all),7,3355468.0,9524793.0,2.84,3.17,0.387
1,Bag Snacks,(all),(all),3,820046.0,1553086.0,1.89,1.99,0.358
2,Bag Snacks,Frito Lay,(all),1,106144.0,304760.0,2.87,3.06,0.273
6,Bag Snacks,Mksl,(all),1,36277.0,75301.0,2.08,2.16,0.207
10,Bag Snacks,Private Label,(all),1,448113.0,642485.0,1.43,1.44,0.328
14,Bag Snacks,Shultz,(all),1,102492.0,184939.0,1.80,2.03,0.462
18,Bag Snacks,Snyder's,(all),1,127020.0,345600.0,2.72,2.95,0.465
22,Cold Cereal,(all),(all),3,1881876.0,5040418.0,2.68,2.99,0.332
23,Cold Cereal,General Mills,(all),1,573089.0,1707538.0,2.98,3.31,0.367
27,Cold Cereal,Kellogg,(all),1,496046.0,1393659.0,2.81,3.29,0.385


## Promotion mix
Share of store-product-weeks by promotion type (BR-02).

In [3]:
s = pd.read_csv('data/clean/sales_promo_weekly.csv')
s['promo_type'].value_counts(normalize=True).round(3)

promo_type
No promo             0.715
TPR only             0.135
Display              0.066
Feature + Display    0.045
Feature              0.040
Name: proportion, dtype: float64

## Base price vs price paid
The base price is the regular shelf price; the gap with the price paid is the discount.

In [4]:
s.groupby('promo_type')[['discount_pct']].describe().round(3)

discount_pct                                                 
                         count   mean    std    min    25%    50%    75%    max
promo_type                                                                     
Display                34291.0  0.124  0.138  0.000  0.000  0.094  0.203  0.799
Feature                20800.0  0.229  0.125  0.000  0.144  0.231  0.319  0.782
Feature + Display      23376.0  0.252  0.113  0.000  0.173  0.261  0.334  0.776
No promo              375541.0  0.000  0.000  0.000  0.000  0.000  0.000  0.000
TPR only               70734.0  0.218  0.105  0.096  0.140  0.188  0.268  1.000

## Quality checks (`sql/05_data_quality.sql`)

In [5]:
pd.read_csv('data/output/sql_05_data_quality.csv')

,check_name,rows_hit,note
0,Rows in weekly sales,524950.0,NaN
1,Missing or non-positive price,209.0,excluded from models
2,Price above base price (+5%),2066.0,excluded from models
3,Extreme price (> 3x or < 1/3 product median),52.0,excluded from models
4,Zero or negative units,5.0,excluded from models
5,Duplicate week x store x product rows,0.0,must be 0
6,Stores listed twice in store lookup,2.0,conflicting segment: first record kept
7,Store not in store lookup,0.0,excluded from market views
8,Store x product pairs with missing weeks,2684.0,possible out-of-stock / delisting
9,Products phasing out (last 13 weeks < 25% of n...,4.0,"kept in history, not simulated"
